In [1]:
import glob
import numpy as np
import pandas as pd
pd.set_option("display.max_columns", 50)

path = glob.glob("data/public_reviews_dataset_cleaned.csv")[0]
df = pd.read_csv(path, low_memory=False)
print(path, df.shape)
print(df.dtypes)
df.head(3)

COL = { 
    "pid": "asin",
    "rid": "reviewer_id",
    "rating": "review_rating",
    "text": "review_text",
    "date": "review_date",
}
LABEL = "fake_review_product"
LEAK = ["reviewer_classified_fake", "reviewer_classified_honest",
         "reviewer_labeled_fake", "reviewer_labeled_honest"]
CAMP = "fake_review_campaign_start_date"
DEL = "review_is_removed_by_amazon"

df[COL["date"]] = pd.to_datetime(df[COL["date"]], errors="coerce")

data/public_reviews_dataset_cleaned.csv (381734, 23)
asin                                   str
review_id                              str
reviewer_id                            str
review_title                           str
review_text                            str
review_rating                        int64
review_date                            str
product_title                          str
product_url                            str
number_of_helpful                  float64
number_of_photos                   float64
photo_thumbnail_urls                   str
photo_fullsize_urls                    str
asin_url                               str
review_url                             str
reviewer_url                           str
fake_review_campaign_start_date        str
fake_review_product                   bool
reviewer_classified_fake              bool
reviewer_classified_honest            bool
reviewer_labeled_fake              float64
reviewer_labeled_honest            float64
r

In [2]:
pid, rat, txt, dt = COL["pid"], COL["rating"], COL["text"], COL["date"]

print("yorum:", len(df), " ürün:", df[pid].nunique(), " yorumcu:", df[COL["rid"]].nunique())
print("tarih aralığı:", df[dt].min(), "->", df[dt].max())

# Etiket dağılımı (yorum düzeyinde ve ürün düzeyinde)
print(df[LABEL].value_counts(dropna=False))
print(df.groupby(pid)[LABEL].first().value_counts(dropna=False))

# Etiket bir ürün içinde sabit mi? (0 çıkmalı)
chk = df.groupby(pid)[LABEL].nunique()
print("etiketi değişen ürün sayısı:", int((chk > 1).sum()))

# Puan dağılımı ve ürün başına yorum sayısı
print(df[rat].value_counts(normalize=True).sort_index().round(3))
print(df.groupby(pid).size().describe().round(1))

# Etiketle ilgili sütunlar: ne kadarı boş, kaç farklı değer var?
for c in LEAK + [CAMP, DEL]:
    print(c, "| boş oranı:", round(df[c].isna().mean(), 3), "| farklı değer:", df[c].nunique(dropna=True))

# Silinme sütunu 15 Haziran 2020'den sonra güvenilmez:
cut = pd.Timestamp("2020-06-15")
print("15 Haz 2020 sonrası yorum oranı:", round((df[dt] >= cut).mean(), 3))

yorum: 381734  ürün: 3389  yorumcu: 334342
tarih aralığı: 2000-06-12 00:00:00 -> 2021-01-19 00:00:00
fake_review_product
False    221181
True     160553
Name: count, dtype: int64
fake_review_product
False    1895
True     1494
Name: count, dtype: int64
etiketi değişen ürün sayısı: 0
review_rating
1    0.289
2    0.117
3    0.140
4    0.278
5    0.176
Name: proportion, dtype: float64
count    3389.0
mean      112.6
std       184.4
min         1.0
25%        17.0
50%        46.0
75%       127.0
max      2251.0
dtype: float64
reviewer_classified_fake | boş oranı: 0.0 | farklı değer: 2
reviewer_classified_honest | boş oranı: 0.0 | farklı değer: 2
reviewer_labeled_fake | boş oranı: 0.79 | farklı değer: 2
reviewer_labeled_honest | boş oranı: 0.79 | farklı değer: 2
fake_review_campaign_start_date | boş oranı: 0.624 | farklı değer: 206
review_is_removed_by_amazon | boş oranı: 0.226 | farklı değer: 2
15 Haz 2020 sonrası yorum oranı: 0.228


In [3]:
g = df.groupby(COL["pid"]).agg(kampanya_var=(CAMP, lambda s: s.notna().any()), etiket=(LABEL, "first"))
pd.crosstab(g["kampanya_var"], g["etiket"])

etiket,False,True
kampanya_var,,
False,1895,45
True,0,1449


In [4]:
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

os.makedirs("results", exist_ok=True)

d = df.dropna(subset=[pid, rat, dt]).copy()
d[txt] = d[txt].fillna("").astype(str)
d[rat] = d[rat].astype(float)

# ---- PUAN özellikleri ----
g = d.groupby(pid)[rat]
R = pd.DataFrame({"r_n": g.size(), "r_mean": g.mean(), "r_std": g.std(ddof=0)})
R["r_share5"]  = (d[rat] == 5).groupby(d[pid]).mean()
R["r_share12"] = (d[rat] <= 2).groupby(d[pid]).mean()
ct = pd.crosstab(d[pid], d[rat], normalize="index")
R["r_entropy"] = -(ct * np.log2(ct.where(ct > 0, 1))).sum(axis=1)

# ---- ZAMAN özellikleri ----
d["week"] = d[dt].dt.to_period("W").dt.start_time
wk = d.groupby([pid, "week"]).size()
T = pd.DataFrame({"t_max_week": wk.groupby(level=0).max(), "t_mean_week": wk.groupby(level=0).mean()})
T["t_burst"] = T["t_max_week"] / T["t_mean_week"]
d["age"] = (d[dt] - d.groupby(pid)[dt].transform("min")).dt.days
T["t_span_days"] = d.groupby(pid)["age"].max()
early = d[d["age"] <= 30]
T["t_early_share"] = (early.groupby(pid).size() / R["r_n"]).reindex(T.index).fillna(0)
T["t_early5"] = (early[rat] == 5).groupby(early[pid]).mean().reindex(T.index)
d["rank_pct"] = d.groupby(pid)[dt].rank(pct=True, method="first")
late_mean  = d[d["rank_pct"] > 0.5].groupby(pid)[rat].mean()
early_mean = d[d["rank_pct"] <= 0.5].groupby(pid)[rat].mean()
T["t_drift"] = (late_mean - early_mean).reindex(T.index)

# ---- METİN özellikleri ----
d["len"] = d[txt].str.split().str.len()
d["excl"] = d[txt].str.count("!")
X_text = d.groupby(pid).agg(x_len_mean=("len", "mean"), x_len_std=("len", "std"), x_excl=("excl", "mean"))
norm = d[txt].str.lower().str.strip()
X_text["x_dup"] = norm.groupby(d[pid]).apply(lambda s: s.duplicated(keep=False).mean())

def mean_sim(s):
    s = s[s.str.len() > 0]
    if len(s) < 3:
        return np.nan
    s = s.sample(min(len(s), 40), random_state=0)
    try:
        M = TfidfVectorizer().fit_transform(s)
    except ValueError:
        return np.nan
    S = cosine_similarity(M)
    n = S.shape[0]
    return (S.sum() - n) / (n * (n - 1))

X_text["x_sim"] = d.groupby(pid)[txt].agg(mean_sim)

# ---- Birleştir ----
X = R.join(T).join(X_text)
y = d.groupby(pid)[LABEL].first().astype(int).loc[X.index]

GROUPS = {
    "puan":  [c for c in X.columns if c.startswith("r_")],
    "zaman": [c for c in X.columns if c.startswith("t_")],
    "metin": [c for c in X.columns if c.startswith("x_")],
}
ALL = sum(GROUPS.values(), [])
X.assign(y=y).to_csv("results/product_table.csv")
print(X.shape, y.value_counts().to_dict())
X.describe().T.round(2)

(3389, 18) {0: 1895, 1: 1494}


,count,mean,std,min,25%,50%,75%,max
r_n,3389.0,112.64,184.41,1.00,17.00,46.00,127.00,2251.00
r_mean,3389.0,3.05,0.79,1.00,2.51,2.96,3.50,5.00
r_std,3389.0,1.27,0.36,0.00,1.18,1.32,1.47,2.00
r_share5,3389.0,0.23,0.26,0.00,0.03,0.10,0.37,1.00
r_share12,3389.0,0.38,0.21,0.00,0.24,0.38,0.51,1.00
r_entropy,3389.0,1.72,0.51,-0.00,1.55,1.91,2.06,2.32
t_max_week,3389.0,6.28,8.04,1.00,2.00,4.00,7.00,123.00
t_mean_week,3389.0,2.17,2.35,1.00,1.18,1.50,2.25,50.02
t_burst,3389.0,2.54,1.14,1.00,1.79,2.47,3.14,23.23
t_span_days,3389.0,734.32,836.60,0.00,269.00,443.00,827.00,7463.00


In [6]:
import lightgbm as lgb
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def mk_lr():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=1000))

def mk_gbm():
    return lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=15,
                              colsample_bytree=0.8, random_state=42, verbose=-1)

def cv_score(make_model, cols, X, y):
    """Her ürün test katında tam bir kez tahmin edilir (oof)."""
    oof = np.zeros(len(y))
    for tr, te in skf.split(X, y):
        m = make_model()
        m.fit(X.iloc[tr][cols], y.iloc[tr])
        oof[te] = m.predict_proba(X.iloc[te][cols])[:, 1]
    s = {"auc": roc_auc_score(y, oof), "pr_auc": average_precision_score(y, oof), "f1": f1_score(y, oof >= 0.5)}
    return oof, s

# Baseline: sadece puan özellikleri + lojistik regresyon
_, base = cv_score(mk_lr, GROUPS["puan"], X, y)
print("baseline:", {k: round(v, 3) for k, v in base.items()})

baseline: {'auc': 0.915, 'pr_auc': 0.915, 'f1': 0.846}


In [7]:
rows = []
sets = {**GROUPS, "hepsi": ALL}
for name, cols in sets.items():
    for mname, mk in [("LogReg", mk_lr), ("LightGBM", mk_gbm)]:
        _, s = cv_score(mk, cols, X, y)
        rows.append({"özellikler": name, "model": mname, **s})
res = pd.DataFrame(rows).round(3)
res.to_csv("results/ablation.csv", index=False)
res

,özellikler,model,auc,pr_auc,f1
0,puan,LogReg,0.915,0.915,0.846
1,puan,LightGBM,0.925,0.931,0.853
2,zaman,LogReg,0.890,0.873,0.777
3,zaman,LightGBM,0.892,0.887,0.782
4,metin,LogReg,0.769,0.708,0.631
5,metin,LightGBM,0.769,0.734,0.654
6,hepsi,LogReg,0.934,0.930,0.853
7,hepsi,LightGBM,0.941,0.945,0.865


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Ürün başına en fazla 100 yorumu tek metin yap
docs = d.groupby(pid)[txt].apply(lambda s: " ".join(s.sample(min(len(s), 100), random_state=0)))
docs = docs.loc[X.index]

oof_tfidf = np.zeros(len(y))
for tr, te in skf.split(docs, y):
    vec = TfidfVectorizer(min_df=3, max_features=20000, ngram_range=(1, 2), sublinear_tf=True)
    A = vec.fit_transform(docs.iloc[tr])      # sözlük SADECE eğitim katından öğrenilir
    B = vec.transform(docs.iloc[te])
    m = LogisticRegression(max_iter=2000).fit(A, y.iloc[tr])
    oof_tfidf[te] = m.predict_proba(B)[:, 1]

print("TF-IDF + LogReg  AUC:", round(roc_auc_score(y, oof_tfidf), 3),
      " PR-AUC:", round(average_precision_score(y, oof_tfidf), 3))

TF-IDF + LogReg  AUC: 0.907  PR-AUC: 0.913


In [9]:
vec = TfidfVectorizer(min_df=3, max_features=20000, ngram_range=(1, 2), sublinear_tf=True)
A = vec.fit_transform(docs)
m = LogisticRegression(max_iter=2000).fit(A, y)
terms = np.array(vec.get_feature_names_out())
order = np.argsort(m.coef_[0])
print("ETİKETSİZ ürünlerde öne çıkan kelimeler:", list(terms[order[:25]]))
print("ETİKETLİ ürünlerde öne çıkan kelimeler:", list(terms[order[-25:]]))

ETİKETSİZ ürünlerde öne çıkan kelimeler: ['but', 'however', 'the only', 'disappointed', 'ok', 'but the', 'not', 'months', 'otherwise', 'would be', 'only', 'wish', 'is that', 'not the', 'christmas', 'except', 'as', 'fine', 'service', 'reason', 'however it', 'maybe', 'but it', 'year', 'the']
ETİKETLİ ürünlerde öne çıkan kelimeler: ['perfectly', 'home', 'and easy', 'very convenient', 'is easy', 'is really', 'it easy', 'is also', 'color', 'very good', 'comes with', 'so easy', 'also', 'it very', 'is perfect', 'amazing', 'highly recommend', 'highly', 'is very', 'perfect', 'convenient', 'easy', 'very easy', 'easy to', 'br']


In [10]:
import re
d_clean = d.copy()
d_clean[txt] = d_clean[txt].fillna("").astype(str).str.replace(r"<[^>]+>", " ", regex=True)
docs2 = d_clean.groupby(pid)[txt].apply(lambda s: " ".join(s.sample(min(len(s), 100), random_state=0)))
docs2 = docs2.loc[X.index]

oof2 = np.zeros(len(y))
for tr, te in skf.split(docs2, y):
    vec2 = TfidfVectorizer(min_df=3, max_features=20000, ngram_range=(1, 2), sublinear_tf=True)
    A2 = vec2.fit_transform(docs2.iloc[tr])
    B2 = vec2.transform(docs2.iloc[te])
    m2 = LogisticRegression(max_iter=2000).fit(A2, y.iloc[tr])
    oof2[te] = m2.predict_proba(B2)[:, 1]

print("HTML temizlendikten sonra TF-IDF AUC:", round(roc_auc_score(y, oof2), 3),
      " PR-AUC:", round(average_precision_score(y, oof2), 3))

HTML temizlendikten sonra TF-IDF AUC: 0.905  PR-AUC: 0.912


In [11]:
# En çok yanlış tahmin edilen 20 ürün (oof = görmediği katta yapılan tahmin)
oof_g, _ = cv_score(mk_gbm, ALL, X, y)
err = pd.Series(np.abs(oof_g - y.values), index=X.index).sort_values(ascending=False).head(20)
out_err = X.loc[err.index, ["r_n", "r_mean", "r_share5", "t_burst", "x_sim"]].round(2)
out_err["gerçek_etiket"] = y.loc[err.index]
out_err["tahmin"] = oof_g[X.index.get_indexer(err.index)].round(2)
out_err

,r_n,r_mean,r_share5,t_burst,x_sim,gerçek_etiket,tahmin
asin,,,,,,,
B081D97V7W,3,4.67,0.67,1.00,0.22,0,1.00
B00PH1MQV8,254,3.03,0.04,2.61,0.09,1,0.00
B003MROHZW,623,2.22,0.01,3.88,0.08,1,0.00
B07WR9R399,27,4.63,0.78,2.22,0.10,0,1.00
B071KCHS1D,6,1.00,0.00,1.00,0.03,1,0.00
B01HCKQG7G,475,2.38,0.03,3.58,0.06,1,0.00
B01IXZGYLM,5,3.40,0.00,1.00,0.07,1,0.00
B013CE67W0,53,3.06,0.04,3.17,0.09,1,0.00
B0749967DB,41,2.59,0.00,1.80,0.06,1,0.00


In [12]:
# Yorumcu düzeyindeki etiketleri ürün düzeyine topla (ürünün yorumcularının kaçı "sahte" işaretli?)
print(df[LEAK[2]].value_counts(dropna=False).head())

rev = df.groupby(pid).agg(
    leak_classified_fake=(LEAK[0], "mean"),
    leak_labeled_fake=(LEAK[2], "mean"),   # etiketsiz yorumcular (boş) hesaba katılmaz
)
X_leak = X.join(rev)

_, clean = cv_score(mk_gbm, ALL, X, y)
_, leaky = cv_score(mk_gbm, ALL + list(rev.columns), X_leak, y)
leak_tab = pd.DataFrame({"temiz": clean, "sızıntılı": leaky}).T.round(3)
leak_tab["fark_auc"] = (leak_tab["auc"] - leak_tab.loc["temiz", "auc"]).round(3)
leak_tab.to_csv("results/leakage.csv")
leak_tab

reviewer_labeled_fake
NaN    301453
0.0     58143
1.0     22138
Name: count, dtype: int64


,auc,pr_auc,f1,fark_auc
temiz,0.941,0.945,0.865,0.000
sızıntılı,0.977,0.981,0.941,0.036


In [14]:
import matplotlib
matplotlib.use("Agg")  # Jupyter'de bu satırı silebilirsin
import matplotlib.pyplot as plt
import shap

# Risk skoru: her ürün için "görmediği" katın tahmini (oof) -> dürüst skor
oof, _ = cv_score(mk_gbm, ALL, X, y)
risk = pd.Series(oof, index=X.index, name="risk")

# Açıklamalar için tüm veriyle eğitilmiş model
final = mk_gbm().fit(X[ALL], y)
explainer = shap.TreeExplainer(final)
sv = explainer.shap_values(X[ALL])
if isinstance(sv, list):          # eski shap sürümleri iki sınıf için liste döndürür
    sv = sv[1]
sv = np.asarray(sv)
if sv.ndim == 3:                  # (n, özellik, sınıf) biçimi
    sv = sv[:, :, 1]

plt.figure()
shap.summary_plot(sv, X[ALL], show=False)
plt.tight_layout()
plt.savefig("results/shap_summary.png", dpi=150)

def reasons(i, k=5):
    idx = np.argsort(-np.abs(sv[i]))[:k]
    parts = []
    for j in idx:
        yon = "artırıyor" if sv[i, j] > 0 else "azaltıyor"
        parts.append(f"{ALL[j]}={X[ALL].iloc[i, j]:.2f} (riski {yon})")
    return " | ".join(parts)

app = pd.DataFrame({
    "product_id": X.index,
    "risk": risk.values,
    "etiket": y.values,
    "nedenler": [reasons(i) for i in range(len(X))],
})
app = app.join(X[["r_n", "r_mean", "r_share5", "t_burst"]], on="product_id")
app.sort_values("risk", ascending=False).to_csv("results/app_data.csv", index=False)
app.sort_values("risk", ascending=False).head()

C:\Users\ASUS\fake-review-risk\.venv\Lib\site-packages\shap\explainers\_tree.py:578: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


,product_id,risk,etiket,nedenler,r_n,r_mean,r_share5,t_burst
2463,B07YHKYMJJ,0.999893,1,r_share5=0.58 (riski artırıyor) | t_mean_week=...,627,3.746411,0.577352,2.894737
2620,B07ZH6TPT9,0.999893,1,r_share5=0.51 (riski artırıyor) | t_span_days=...,102,3.617647,0.509804,2.803922
2553,B07Z3FG46V,0.999889,1,r_share5=0.46 (riski artırıyor) | t_mean_week=...,668,3.396707,0.462575,2.595808
3035,B083DL9WTK,0.999879,1,r_share5=0.53 (riski artırıyor) | r_std=1.76 (...,396,3.479798,0.532828,3.209596
3316,B0869MCRPS,0.999877,1,r_share5=0.65 (riski artırıyor) | t_span_days=...,60,3.783333,0.650000,3.300000


In [15]:
from scipy.stats import spearmanr

# Sadece silinme bilgisinin güvenilir olduğu dönem: 15 Haziran 2020 öncesi
cut = pd.Timestamp("2020-06-15")
old = df[df[dt] < cut].copy()
old[DEL] = old[DEL].astype(float)
del_rate = old.groupby(pid)[DEL].mean()

tab = pd.DataFrame({"risk": risk, "silinme_orani": del_rate}).dropna()
tab["dilim"] = pd.qcut(tab["risk"].rank(method="first"), 5,
                       labels=["1 (en düşük)", "2", "3", "4", "5 (en yüksek)"])
out = tab.groupby("dilim", observed=True)["silinme_orani"].agg(["mean", "count"]).round(3)
rho, p = spearmanr(tab["risk"], tab["silinme_orani"])
print(out)
print("Spearman rho =", round(rho, 3), " p =", round(p, 4))
out.to_csv("results/dogrulama_silinme.csv")

                mean  count
dilim                      
1 (en düşük)   0.020    667
2              0.030    667
3              0.066    666
4              0.235    667
5 (en yüksek)  0.361    667
Spearman rho = 0.611  p = 0.0
